# Purpose: Find out how to classify probe movements

# Setup & load

In [ ]:
import numpy as np                                                                                                                                                                                                                                         
import matplotlib.pyplot as plt                                                                                                                                                                                                                            
import mapd     
from mapd.kinematics import _rolling_rms, _classify_states_schmitt, velocity, STATE_REST, STATE_DRIFT, STATE_MOVE
%matplotlib inline
%load_ext autoreload   
%autoreload 2

In [ ]:
T = mapd.Table('LEDFlashTriggerPiezoControl_250304_F3_C1_Table.parquet')
T.df.head()

In [ ]:
import pandas as pd
import mapd.kinematics as kin
# from mapd.kinematics import (
#     _build_trace, detect_movement_bouts, detect_bouts_across_trials, plot_bouts,
# )

T.find_successful_trials()

# Pick a hard-success trial to inspect (change trial number here)
hard_rows = T.df.loc[T.df['hard_success']].copy()
hard_rows['next_trial'] = T.df['Trial'].shift(-1).loc[hard_rows.index]
print(f"{len(hard_rows)} hard-success trials")
# hard_rows.index.to_list()

In [ ]:
# Inspect a single trial — change the index in hard_rows.iloc[] to browse
# row = hard_rows.iloc[0]
for stn in hard_rows.index:
    row = hard_rows.loc[stn]
    t_num = row.name
    trial1, trial2 = row['Trial'], row['next_trial']

    v_th = 60
    v_rest = 30

    bouts, states, t_b, x_b = kin.detect_bouts_across_trials(
        [trial1, trial2], v_th=v_th, v_rest=v_rest)

    fig = kin.plot_bouts(bouts, states, t_b, x_b,
                    trials=[trial1, trial2],
                    v_th=v_th, v_rest=v_rest,
                    # title=f'Trial {t_num} (hard success)',
                    show_grid=False,
                    show_drift_patches=True,
                    cum_vrms_all_bouts=True)
    # fig.show()
    OUT_HTML =  f'trial_{stn}_{T.dfc}_tune_params.html'
    fig.write_html(OUT_HTML)

In [ ]:
bouts